In [ ]:
from collatex import *
from lxml import etree
import json,re,os,itertools

In [ ]:
def splitId(id):
    """Splits @id value like 4008a into parts, for sorting"""
    linenoRegex = re.compile('(\d+)(.*)')
    results = linenoRegex.match(id).groups()
    return (int(results[0]),results[1])

In [ ]:
class WitnessSet:
    def __init__(self,witnessList):
        self.witnessList = witnessList
    def all_witnesses(self):
        """List of tuples consisting of siglum and contents"""
        return [Witness(witness) for witness in self.witnessList]
    def all_ids(self):
        """Sorted deduplicated list of all ids in corpus"""
        return sorted(set(itertools.chain.from_iterable([witness.XML().xpath('//l/@id') for witness in self.all_witnesses()])),key=splitId)
    def get_lines_by_id(self,id):
        """List of tuples of siglum plus <l> element from each witness that corresponds to a certain line"""
        witnesses_with_line = []
        for witness in self.all_witnesses():
            try:
                witnesses_with_line.append((witness.siglum,witness.XML().xpath('//l[@id = ' + id + ']')[0]))
            except:
                pass
        return witnesses_with_line
    def generate_json_input(self, lineId):
        """JSON input to CollateX for an <l> segment"""
        json_input = {}
        witnesses = []
        for witness in self.get_lines_by_id(lineId):
            currentWitness = {}
            currentWitness['id'] = witness[0]
            currentWitness['tokens'] = Line(witness[1]).tokens()
            witnesses.append(currentWitness)
        json_input['witnesses'] = witnesses
        return json_input

In [ ]:
class Witness:
    """Each witness in the witness set is an instance of class Witness"""
    def __init__(self,witness):
        self.witness = witness
        self.siglum = self.witness[0]
        self.contents = self.witness[1]
    def XML(self):
        return etree.XML(self.contents)

In [ ]:
class Line:
    """An instance of Line is a line in a witness, expressed as an <l> element"""
    addWMilestones = etree.XML("""
    <xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
        <xsl:output method="xml" indent="no" encoding="UTF-8" omit-xml-declaration="yes"/>
        <xsl:template match="*|@*">
            <xsl:copy>
                <xsl:apply-templates select="node() | @*"/>
            </xsl:copy>
        </xsl:template>
        <xsl:template match="/*">
            <xsl:copy>
                <xsl:apply-templates select="@*"/>
                <!-- insert a <w/> milestone before the first word -->
                <w/>
                <xsl:apply-templates/>
            </xsl:copy>
        </xsl:template>
        <!-- convert <add>, <sic>, and <crease> to milestones (and leave them that way)
             CUSTOMIZE HERE: add other elements that may span multiple word tokens
        -->
        <xsl:template match="add | del | subst | abbr | expan | choice | orig | reg | pb">
            <xsl:element name="{name()}">
                <xsl:attribute name="n">start</xsl:attribute>
            </xsl:element>
            <xsl:apply-templates/>
            <xsl:element name="{name()}">
                <xsl:attribute name="n">end</xsl:attribute>
            </xsl:element>
        </xsl:template>
        <xsl:template match="note"/>
        <xsl:template match="text()">
            <xsl:call-template name="whiteSpace">
                <xsl:with-param name="input" select="translate(.,'&#x0a;',' ')"/>
            </xsl:call-template>
        </xsl:template>
        <xsl:template name="whiteSpace">
            <xsl:param name="input"/>
            <xsl:choose>
                <xsl:when test="not(contains($input, ' '))">
                    <xsl:value-of select="$input"/>
                </xsl:when>
                <xsl:when test="starts-with($input,' ')">
                    <xsl:call-template name="whiteSpace">
                        <xsl:with-param name="input" select="substring($input,2)"/>
                    </xsl:call-template>
                </xsl:when>
                <xsl:otherwise>
                    <xsl:value-of select="substring-before($input, ' ')"/>
                    <w/>
                    <xsl:call-template name="whiteSpace">
                        <xsl:with-param name="input" select="substring-after($input,' ')"/>
                    </xsl:call-template>
                </xsl:otherwise>
            </xsl:choose>
        </xsl:template>
    </xsl:stylesheet>
    """)
    transformAddW = etree.XSLT(addWMilestones)
    xsltWrapW = etree.XML('''
    <xsl:stylesheet xmlns:xsl="http://www.w3.org/1999/XSL/Transform" version="1.0">
        <xsl:output method="xml" indent="no" omit-xml-declaration="yes"/>
        <xsl:template match="/*">
            <xsl:copy>
                <xsl:apply-templates select="w"/>
            </xsl:copy>
        </xsl:template>
        <xsl:template match="w">
            <!-- faking <xsl:for-each-group> as well as the "<<" and except" operators -->
            <xsl:variable name="tooFar" select="following-sibling::w[1] | following-sibling::w[1]/following::node()"/>
            <w>
                <xsl:copy-of select="following-sibling::node()[count(. | $tooFar) != count($tooFar)]"/>
            </w>
        </xsl:template>
    </xsl:stylesheet>
    ''')
    transformWrapW = etree.XSLT(xsltWrapW)
    def __init__(self,line):
        self.line = line
    def tokens(self):
        return [Word(token).createToken() for token in Line.transformWrapW(Line.transformAddW(self.line)).xpath('//w')]
        

In [ ]:
class Word:
    unwrapRegex = re.compile('<w>(.*)</w>')
    stripTagsRegex = re.compile('<.*?>')
    def __init__(self,word):
        self.word = word
    def unwrap(self):
        return Word.unwrapRegex.match(etree.tostring(self.word,encoding='unicode')).group(1)
    def normalize(self):
        return Word.stripTagsRegex.sub('',self.unwrap().lower())
    def createToken(self):
        token = {}
        token['t'] = self.unwrap()
        token['n'] = self.normalize()
        return token

In [ ]:
os.listdir('cliges8')

In [ ]:
Wit=WitnessSet([(inputFile[0],open('cliges8/' + inputFile,'rb').read()) for inputFile in os.listdir('cliges8')])

In [ ]:
Wit1 = Wit.generate_json_input('98')
Wit1

In [ ]:
### ici pour la création de la boucle 

#création d'une liste vide dans laquelle on va mettre tous les résultats successifs
listeDesColl = []
#la boucle marche
i=1
#donc ici il faudrait améliorer la boucle pour que ça fasse tous tes vers
for i in range(1,99):
        print(i)
        j=str(i)
        #il fallait juste faire ça, je m'étais trompée dans l'ordre des guillemets, je crois bien...
        w = Wit.generate_json_input("'"+j+"'")
        print(w)
        #segmentation=False pour avoir mot à mot LI
        collationText = collate(w,output='table', segmentation=False)
        print(collationText)
        collationJSON = collate(w,output='json', segmentation=False)
        print(collationJSON)
        collationHTML2 = collate(w,output='html2', segmentation=False)
        #ensuite XML (ou TEI)
        collationXML = collate(w, output="xml", segmentation=False)
        print(collationXML)
        #on va créer artificiellement un élément l qui va avoir le numéro du vers
        doc = "<l n='" + j +"'>" + collationXML + "</l>"
        #on remplit la liste
        listeDesColl.append(doc)

In [ ]:
#''.join pour ne pas avoir un format liste, mais le texte en entier
print(''.join(listeDesColl))

In [ ]:
### ici pour la création de la boucle 

#création d'une liste vide dans laquelle on va mettre tous les résultats successifs
listeDesColl = []
#la boucle marche
i=1
#donc ici il faudrait améliorer la boucle pour que ça fasse tous tes vers
for i in range(1,99):
        print(i)
        j=str(i)
        #il fallait juste faire ça, je m'étais trompée dans l'ordre des guillemets, je crois bien...
        w = Wit.generate_json_input("'"+j+"'")
        print(w)
        #segmentation=False pour avoir mot à mot LI
        collationHTML2 = collate(w,output='html2', segmentation=False)
        #on va créer artificiellement un élément l qui va avoir le numéro du vers
        doc = "<l n='" + j +"'>" + collationXML + "</l>"
        #on remplit la liste
        listeDesColl.append(doc)

In [ ]:
### ici pour la création de la boucle 

#création d'une liste vide dans laquelle on va mettre tous les résultats successifs
listeDesColl = []
#la boucle marche
i=1
#donc ici il faudrait améliorer la boucle pour que ça fasse tous tes vers
for i in range(1,99):
        print(i)
        j=str(i)
        #il fallait juste faire ça...
        w = Wit.generate_json_input("'"+j+"'")
        #on vire print(w) pour n'avoir que lq sortie txt
        #segmentation=False pour avoir mot à mot LI
        collationText = collate(w,output='table', segmentation=False)
        print(collationText)